# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramithnayak8/ML_pipeline/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This is a **ranking / scoring** task, built on top of a binary classifier. The deliverable isn't
a single yes/no verdict on one page — it's an ordered queue an editor works down under a fixed
review capacity. Under the hood, the reference pipeline trains a classifier to predict
`is_declining_label`, then uses its predicted probability as the ranking score, blended with the
transparent baseline score into one final priority number. This matches the framing skill's own
mapping directly: "Which ones first?" -> Ranking/scoring, target a priority score, metric
Precision@K.

In [1]:
import json

res = json.load(open("../../outputs/model_results.json"))
print("Target the classifier predicts:", res["target"])
print("How that score gets used:", res["prediction_output"],
      "-> blended into a ranked queue, never read as a single verdict")
print(f"Split strategy: {res['split_strategy']} "
      f"({res['train_rows']:,} train / {res['test_rows']:,} test rows)")


Target the classifier predicts: is_declining_label
How that score gets used: data\processed\model_predictions.csv -> blended into a ranked queue, never read as a single verdict
Split strategy: client_holdout (27,675 train / 2,325 test rows)


## 2. Target or proxy

Current target: `is_declining_label = (trend_direction == "down")`. `trend_direction` is a bucket
computed from `trend_pct`, a real measured percentage change over the trailing 90 days — so the
underlying number is observed, but the label describes the **current** window, not a **future**
outcome. That makes it a proxy label, exactly as this repo's own guide names it: good enough to
prove the workflow end to end, not the strongest possible capstone target.

A stronger label, which I plan to move toward once I've explored the warehouse release:
`features from the prior 90 days -> decline (or recovery) over the NEXT 30 days`, built from
`fact_content_daily_performance`'s daily grain. That shift turns "is this page currently down"
into "will this page keep sliding" — the question an editor actually needs answered before
spending time on a page.

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df[["trend_pct", "trend_direction", "is_declining_label"]].head(5).to_string(index=False))
print()
print(f"positive rate: {df['is_declining_label'].mean():.1%}  "
      f"(matches outputs/model_results.json: {res['target_positive_rate']:.1%})")


 trend_pct trend_direction  is_declining_label
     -41.4            down                   1
     -57.7            down                   1
     -60.9            down                   1
     -13.8          stable                   0
     -34.7            down                   1

positive rate: 54.2%  (matches outputs/model_results.json: 54.2%)


## 3. Success metric

**Precision@50.** An editor works roughly the top 50 items in the ranked queue each cycle — the
same capacity the reference pipeline reports against. Precision@50 answers exactly the question
that matters for this decision: *of the top 50 pages we tell someone to look at, how many are
actually declining?* It is easy to defend to a non-technical stakeholder and it directly measures
the queue's usefulness under a fixed review capacity — unlike plain accuracy or ROC-AUC, which
don't reflect a capacity-constrained decision at all.

I'll also watch Precision@20 (a stricter top slice) and average precision (a whole-ranking view)
as secondary checks, but Precision@50 is the one number I'm willing to be judged on.

In [3]:
print(f"{'baseline rule':>18}: P@20={res['baseline']['baseline_precision_at_20']:.3f}  "
      f"P@50={res['baseline']['baseline_precision_at_50']:.3f}  "
      f"AP={res['baseline']['baseline_average_precision']:.3f}")

for name, m in res["models"].items():
    print(f"{name:>18}: P@20={m['precision_at_20']:.3f}  "
          f"P@50={m['precision_at_50']:.3f}  AP={m['average_precision']:.3f}")


     baseline rule: P@20=0.150  P@50=0.240  AP=0.468
     decision_tree: P@20=0.550  P@50=0.620  AP=0.575
logistic_regression: P@20=0.350  P@50=0.400  AP=0.522
     random_forest: P@20=0.700  P@50=0.680  AP=0.610


## 4. The unit of analysis, as a real dataframe

**One row = one content page**, summarized over its trailing 90 days. NOT one row per client (32
clients each own many pages) and NOT one row per day (that's the warehouse's daily fact table,
a different grain entirely — see `docs/ml-intern-dataset-and-lane-guide.md`). Every score, rank,
and action in this lane attaches to a single `content_id`.

In [4]:
print("content_id unique per row:", df["content_id"].is_unique)
print(f"{len(df):,} rows, {df['client_id'].nunique()} clients, "
      f"{df['content_id'].nunique():,} unique content_ids")

df[["content_id", "client_id", "impressions_90d", "avg_position", "ctr",
    "trend_direction"]].head(5)


content_id unique per row: True
30,000 rows, 32 clients, 30,000 unique content_ids


,content_id,client_id,impressions_90d,avg_position,ctr,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,down
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,down
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,down


## 5. Why ML beats a fixed rule here

The signal isn't concentrated in one or two clean thresholds — it's spread thin across many
weakly-informative, interacting fields. The reference random forest's own top-10 feature
importances (below, read straight from `outputs/model_report.md`) top out at 0.16 and taper off
gradually, with no single dominant field. A hand-written if-statement can reasonably encode two
or three conditions; it can't encode the shifting, second-order combinations of ten-plus fields
built from 52 engineered features. That gap is exactly why the baseline rule — a fixed weighted
formula over 4 sub-scores — tops out at Precision@50 = 0.240, while a model that can learn
interactions reaches 0.680, a ~2.8x lift on the same data (cell 6 above).

In [5]:
report = open("../../outputs/model_report.md", encoding="utf-8").read()
top_features_section = report.split("## Top Features")[1].split("## ")[0]
print(top_features_section.strip())

print()
print(f"Total engineered features the model sees: {res['feature_count']} "
      f"(from {len(res['model_numeric_features'])} numeric + "
      f"{len(res['model_categorical_features'])} categorical raw fields)")


- `days_with_impressions`: 0.1606
- `log_impressions_90d`: 0.1285
- `avg_position`: 0.1084
- `content_age_days`: 0.0950
- `word_count`: 0.0412
- `char_count`: 0.0398
- `ctr`: 0.0332
- `log_clicks_90d`: 0.0325
- `scroll_rate`: 0.0309
- `days_with_sessions`: 0.0299

Total engineered features the model sees: 52 (from 18 numeric + 8 categorical raw fields)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.